<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **3. Global Navigation Satellite Systems**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 3.1 Introduction

Satellite navigation determines position and time from radio signals broadcast by satellites. A receiver measures signal timing, obtains satellite orbit information, and combines observations from several satellites to estimate its own location.

This notebook introduces Global Navigation Satellite Systems (GNSS), using GPS as the main example. It connects orbit geometry, distance measurements, and receiver clock bias, then demonstrates how a known spreading sequence helps recover data from a noisy signal.

Positioning accuracy depends on satellite visibility, geometry, atmospheric effects, reflections, and receiver quality. Meter-level positioning is common under suitable conditions; centimeter-level results require specialized measurements and processing. Reception inside buildings or among tall buildings can be poor [\[03\]](#ref03).

<center>
  <img src="../../.admin/notebooks/picture/gnss_overview.png" 
       alt="Four navigation satellites send one-way signals to a receiver that estimates position and clock bias." 
       width="60%"
      style="height:auto;">
</center>


### 3.2 Global Navigation Satellite Systems

GNSS is the general term for satellite systems providing positioning, navigation, and timing. GPS is the United States' system; it is one member of this broader family. Receivers can combine several constellations to increase the number of available satellite observations. More observations can improve a solution, but their quality and geometry still matter [\[01\]](#ref01).

The following table summarizes constellation types rather than current operational satellite counts. Counts change as satellites enter service, undergo maintenance, or retire.

| System | Country or organization | Orbit arrangement |
|---|---|---|
| GPS | United States | Medium Earth orbit (MEO); a baseline of 24 slots, with additional satellites in operation |
| GLONASS | Russia | MEO |
| Galileo | European Union | MEO |
| BeiDou | China | MEO plus geostationary and inclined geosynchronous orbits |

GPS's baseline arrangement is a design reference, not a live inventory [\[04\]](#ref04). The constellations also differ in signals, time scales, and reference frames; combining them requires receiver support [\[01\]](#ref01).


Medium Earth orbit provides a compromise between a large coverage footprint and changing satellite geometry. GPS satellites orbit at about 20,200 km altitude, completing approximately two orbits per day [\[04\]](#ref04).

A geostationary satellite follows a circular equatorial orbit at about 35,786 km altitude. Its period is one sidereal day, approximately 23 hours 56 minutes, so it appears fixed from the ground. Its high altitude increases propagation time, and its equatorial position gives poor viewing geometry at high latitudes. A single overhead ground-to-satellite path takes about 119 ms; a ground-to-satellite-to-ground path takes at least about 239 ms, before processing and routing delays [\[05\]](#ref05).

Low Earth orbit (LEO) has shorter signal paths and faster apparent satellite motion. Continuous service therefore requires a suitable constellation and handovers. End-to-end network latency includes more than the propagation time and cannot be inferred from altitude alone.

<center>
  <a href="https://upload.wikimedia.org/wikipedia/commons/b/b4/Comparison_satellite_navigation_orbits.svg" target="_blank">
    <img src="../../.admin/notebooks/picture/Comparison_satellite_navigation_orbits.svg" 
         alt="Orbits" 
         width="50%" 
         style="filter: grayscale(20%);">
  </a>
  <p align="center">
    <em>Figure: Satellite Navigation Orbits
    [<a href="https://en.wikipedia.org/wiki/File:Comparison_satellite_navigation_orbits.svg">Source</a>]
    licensed under the
    <a href="https://creativecommons.org/licenses/by-sa/3.0/deed.en">Creative Commons Attribution-Share Alike 3.0 Unported license</a>. Click on the image for an interactive version, with links to the Wikipedia explanations.
    </em>
  </p>
</center>

Earth's radiation belts contain trapped energetic particles that can affect satellite electronics. Their extent and intensity vary, so simple altitude boundaries do not define universally safe orbits. MEO and GEO missions need to account for radiation exposure; LEO missions also face radiation hazards [\[06\]](#ref06).

Operational lifetime depends on fuel, hardware reliability, radiation exposure, and mission design. Atmospheric drag also affects low orbits. An operational lifetime and the time required for an uncontrolled orbit to decay are different quantities.

Research has explored using Starlink communication signals for positioning. Such signals are signals of opportunity, rather than a replacement for the dedicated ranging signals and navigation messages discussed here [\[02\]](#ref02).


### 3.3 Position and receiver clock bias

In an ideal synchronized system, distance equals propagation speed multiplied by travel time:

$$
d=c(t_{\mathrm{receive}}-t_{\mathrm{transmit}}),\qquad c\approx3\times10^8\ \mathrm{m/s}.
$$

A GNSS receiver measures code timing by comparing an incoming signal with a local replica. The navigation message provides timing and orbit information needed to interpret that measurement. The result is a pseudorange: an apparent distance that includes clock offsets and propagation errors [\[07\]](#ref07).

A receiver clock offset of 1 ms contributes about 300 km to a pseudorange measurement. This is a range bias, not automatically a 300 km position error. The receiver estimates its clock bias together with its position.

After applying satellite clock corrections, a simplified measurement equation for satellite $i$ is

$$
\rho_i=\lVert\mathbf{x}-\mathbf{s}_i\rVert+c\,\delta t+\varepsilon_i.
$$

Here $\rho_i$ is the measured pseudorange, $\mathbf{s}_i$ is the satellite position at transmission, $\mathbf{x}$ is the receiver position, $\delta t$ is its clock bias, and $\varepsilon_i$ collects remaining errors. The four basic unknowns are three position coordinates and one clock bias. At least four independent satellite observations with suitable geometry are needed for this unconstrained 3D solution [\[01\]](#ref01), [\[07\]](#ref07).

All measurements contribute to all four unknowns. Additional observations allow an overdetermined solution and can improve robustness. Combining constellations may also require handling differences between their time scales.

<center>
<img src="../../.admin/notebooks/picture/trilateration_gif.gif" 
     alt="Trilateration GIF" 
     width="60%" 
     style="filter: grayscale(40%);">
<p align="center"><em>Geometric illustration with three satellites; a practical 3D solution also estimates receiver clock bias. Source: Advanced Navigation</em></p>
</center>

Code timing, satellite geometry, orbit and clock errors, atmospheric delays, and reflected signals all affect positioning. A wider signal bandwidth can improve timing precision, but bandwidth alone does not guarantee a particular position accuracy [\[03\]](#ref03).

Real-Time Kinematic (RTK) positioning combines carrier-phase measurements with reference-station information. Precise Point Positioning (PPP) uses precise orbit and clock products with receiver measurements. High precision depends on corrections, ambiguity handling, and suitable observations; it is not simply a consequence of wider-band signals [\[07\]](#ref07).


### 3.4 GPS signals and receiver processing

For ordinary positioning, the user's GPS receiver listens to satellite broadcasts without sending a reply. The constellation is designed to provide global coverage, but buildings, terrain, and antenna placement can obstruct signals [\[03\]](#ref03), [\[04\]](#ref04).

| GPS band | Carrier frequency | Example civil signal |
|---|---|---|
| L1 | 1575.42 MHz | C/A; modernized L1C on supporting satellites |
| L2 | 1227.60 MHz | L2C on supporting satellites |
| L5 | 1176.45 MHz | L5 on supporting satellites |

These bands carry different signal components, and not every satellite or receiver supports every signal. The following explanation focuses on legacy GPS L1 C/A [\[08\]](#ref08).

The legacy L1 C/A navigation message runs at 50 bit/s. It provides time information, satellite health, and orbit parameters. Modern GPS signals use other message formats, so this rate should not be applied to every GPS signal [\[08\]](#ref08).

| Information | Purpose |
|---|---|
| Ephemeris | Parameters for calculating an individual satellite's orbit accurately over a limited interval |
| Almanac | Coarser constellation information that can help a receiver find satellites |
| Clock corrections and health | Information for correcting satellite timing and deciding whether a signal should be used |

The receiver combines orbit information with measured signal timing to calculate a position. Reading the navigation message and estimating code timing are related but distinct parts of reception [\[01\]](#ref01).

PRN stands for pseudo-random noise. A PRN code is a deterministic sequence that looks random but can be reproduced by the receiver. A chip is one element of this spreading code. Chips change faster than navigation-data bits. The GPS L1 C/A code contains 1,023 chips, transmitted at 1.023 million chips per second, and repeats every millisecond. The receiver generates a local copy of the satellite’s assigned code [\[08\]](#ref08).

Different satellite codes allow signals to share a carrier frequency, an approach called code division multiple access (CDMA). A receiver compares the incoming waveform with a local code at different alignments. A strong correlation indicates a likely match. Real acquisition also searches frequency offsets caused by motion and clock differences; tracking then refines the estimates [\[01\]](#ref01).


### 3.5 Demonstration: spreading and recovering a message

This simplified direct-sequence spread-spectrum model illustrates modulation, code alignment, and data recovery. It uses six alternating data symbols and a seeded random spreading sequence. It does not generate an actual GPS C/A code, decode a navigation message, measure satellite travel time, or solve for position.

The display uses a 100 Hertz carrier, 100 chips/s, 10 data symbols/s, and 10,000 samples/s. These deliberately slow values make the waveforms readable. Each data symbol contains ten chips; each chip lasts 10 ms. They are demonstration settings, separate from the GPS specifications above.


#### 3.5.1 Transmitter

The plots show the data symbols, spreading sequence, their product, the carrier, and the resulting modulated signal. Binary phase-shift keying (BPSK) represents a sign change as a 180-degree carrier-phase change.


In [ ]:
from pathlib import Path
import runpy

helper = next(p for p in (
    Path("../../.admin/notebooks/gnss_signals.py"),
    Path(".admin/notebooks/gnss_signals.py"),
) if p.is_file())
gnss_demo = runpy.run_path(str(helper.resolve()))
transmitter_result = gnss_demo["show_transmitter"](seed=0)


#### 3.5.2 Receiver

The receiver example adds a spreading-code offset, carrier-phase offset, and additive white Gaussian noise (AWGN). SNR is the signal-to-noise power ratio; 0 dB means equal average signal and noise powers in this simulation.

Carrier removal multiplies the received waveform by a local cosine. The receiver then tests 100 sample offsets within one chip, selects the largest correlation magnitude over the first data symbol, and sums the despread samples for each symbol. The sign of each sum gives the recovered data value.

Only the spreading sequence is shifted, using circular indexing. Symbol boundaries and nominal carrier frequency are known. This is a restricted alignment experiment, not a physical propagation-delay model or a complete acquisition and tracking receiver. Keep the frequency offset at zero for the initial tests.


In [ ]:
snr_db = 0.0
code_phase_offset = 12  # Samples, from 0 to 99
carrier_phase = 0.4     # Radians
freq_offset_hz = 0.0    # Hertz
seed = 0

gnss_demo = runpy.run_path(str(helper.resolve()))
receiver_result = gnss_demo["show_receiver"](
    snr_db=snr_db,
    code_phase_offset=code_phase_offset,
    carrier_phase=carrier_phase,
    freq_offset_hz=freq_offset_hz,
    seed=seed,
)


The correlation plot compares candidate code offsets; its peak is the selected alignment. The final plot compares the original and recovered symbols. Bit error rate (BER) is the fraction of these six symbols recovered incorrectly.

Summing aligned samples can recover a known pattern even when individual samples are noisy. A successful six-symbol run does not establish reliable operation at that SNR. Changing the seed changes both the spreading sequence and noise realization, helping reveal this limitation.


#### 3.5.3 Measuring a code offset, even below the noise

Correlation slides a local code past the received samples and measures their agreement at each candidate offset. When the patterns line up, matching signs reinforce each other and produce a peak. For a known code $p[n]$ and received samples $r[n]$, this example calculates

$$
R[k]=\frac{1}{N}\sum_{n=0}^{N-1}r[n]p[(n-k)\bmod N],\qquad
\hat{k}=\operatorname*{arg\,max}_k |R[k]|.
$$

The peak's location estimates the injected offset. At a sample rate $f_s$, the corresponding time shift is $\hat{\tau}=\hat{k}/f_s$. The example uses a separate 1,023-element random code, one sample per chip, and $f_s=10{,}000$ Hertz. It is not the GPS C/A code. The shift is circular, so it is measured modulo one code period; it is not an absolute satellite travel-time measurement.

The lower plots compare correlation without and with noise. The default SNR is −15 dB: noise power is about 32 times signal power, so the received waveform looks noisy. Nevertheless, the known code can produce a recognizable correlation peak. At the correct alignment, the signal contributions add coherently; independent, zero-mean noise contributions partly cancel. Before normalization, signal amplitude grows with $N$, while noise standard deviation grows with $\sqrt{N}$. This integration improves the correlation output's signal-to-noise ratio.

This works only within limits. The model assumes known carrier timing, independent noise, and no data-bit changes during integration. Very strong noise, frequency errors, or an incorrect code can hide the peak or produce a false maximum. Try changing `snr_db` or `seed` and compare the estimated offset with the injected one. Recovering this offset does not by itself establish a reliable GPS position.


In [ ]:
offset_samples = 137
snr_db = -15.0
seed = 0

gnss_demo = runpy.run_path(str(helper.resolve()))
correlation_result = gnss_demo["show_code_correlation"](
    offset_samples=offset_samples, snr_db=snr_db, seed=seed,
)


### 3.6 Literature

The [GNSS Satellite Navigation collection](https://gitlab.ethz.ch/wireless/BackgroundReading/-/tree/main/GNSS%20Satellite%20Navigation) contains the two textbooks linked below. Reference [\[07\]](#ref07) provides an introduction to observations and positioning algorithms; reference [\[01\]](#ref01) offers a more detailed mathematical treatment.

[<a id="ref01">01</a>] Xu, G., & Xu, Y. (2016). [**GPS: Theory, Algorithms, and Applications (3rd ed.)**](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/GNSS%20Satellite%20Navigation/2016-gps-theory-algorithms-applications-3rd-edition.pdf). Springer, Berlin, Heidelberg. PDF in the BackgroundReading repository.

[<a id="ref02">02</a>] Kozhaya, S., Saroufim, J., & Kassas, Z. M. (2025). [**Unveiling Starlink for PNT**](https://doi.org/10.33012/navi.685). NAVIGATION: Journal of the Institute of Navigation, 72(1), navi.685.



[<a id="ref03">03</a>] GPS.gov. [GPS Accuracy](https://www.gps.gov/gps-accuracy).

[<a id="ref04">04</a>] GPS.gov. [Space Segment](https://www.gps.gov/space-segment).

[<a id="ref05">05</a>] European Space Agency. [Types of orbits](https://www.esa.int/Enabling_Support/Space_Transportation/Types_of_orbits).

[<a id="ref06">06</a>] NASA. [Radiation Belts with Satellites](https://www.nasa.gov/image-article/radiation-belts-with-satellites/).

[<a id="ref07">07</a>] Sanz Subirana, J., Juan Zornoza, J. M., & Hernández-Pajares, M. (2013). [GNSS Data Processing, Volume I: Fundamentals and Algorithms](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/GNSS%20Satellite%20Navigation/2013-esa-gnss-data-processing-volume-1.pdf). ESA TM-23/1.

[<a id="ref08">08</a>] European Space Agency, Navipedia. [GPS Signal Plan](https://gssc.esa.int/navipedia/index.php/GPS_Signal_Plan).



----
### 3.7 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**How would using a geostationary satellite instead of a satellite in medium Earth orbit (MEO) change its apparent motion in the sky, signal travel time, and visibility at high latitudes?**

Answer:


**What is the minimum number of satellites needed to estimate a receiver’s three-dimensional position and clock bias, assuming suitable satellite geometry and no additional constraints?**

Answer:


**What is the ratio of signal power to noise power at SNR values of 5 dB, 0 dB, and −100 dB? For each value, state whether the signal power is greater than, equal to, or less than the noise power.**

Answer:



**In section 3.5.2, set `snr_db` to `5`, `0`, and `-100`, keeping `seed = 0` and the other settings unchanged. Rerun the cell for each value. At which SNR values are all six symbols recovered correctly?**

Answer:


**OPTIONAL: Are all explanations and exercises in this notebook clear? Which parts, if any, need clarification or improvement?**

Answer:
